# VectorEnv Step Benchmark

Compare `SyncVectorEnv` and `AsyncVectorEnv` for SolarSystemLander environment stepping. This benchmark is intentionally separated from `StudyRunner`, dashboard, replay buffer, and DQN training so the signal stays clean.

## Setup

In [ ]:
# cell: setup

from pathlib import Path
import os
import platform
import subprocess
import sys
import time

if not Path("hpo/src").exists():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the rl_lab repository root or in Colab.")
    subprocess.run(["git", "clone", "https://github.com/miwehle/rl_lab.git"], check=True)
    os.chdir("rl_lab")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "hpo/requirements.txt"],
    check=True,
)

for _path in ("dqn/src", "hpo/src"):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import gymnasium as gym
import torch
from gymnasium.vector import AsyncVectorEnv

from hpo.solar_system_lander.environment import DEFAULT_WORLD_MIX, EnvFactory


def gpu_name():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        ).strip()
    except Exception:
        return "no nvidia-smi"


print("python:", platform.python_version())
print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", gpu_name())

## CPU Info

In [ ]:
# cell: cpu-info; requires: setup

CPU_COUNT = os.cpu_count() or 2

try:
    import psutil
except ImportError:
    psutil = None

print("logical cpu_count:", CPU_COUNT)

if psutil is not None:
    print("physical cores:", psutil.cpu_count(logical=False))
    print("logical cores:", psutil.cpu_count(logical=True))

NUM_ENVS_LIST = sorted({5, CPU_COUNT, 2 * CPU_COUNT, 4 * CPU_COUNT})
print("NUM_ENVS_LIST:", NUM_ENVS_LIST)

## Benchmark Config

In [ ]:
# cell: benchmark-config; requires: setup

OBSERVATION_MODE = "10d"
WORLD_MIX = DEFAULT_WORLD_MIX
PARAMS = {}

SEED = 123
WARMUP_STEPS = 50
MEASURE_STEPS = 500
REPEATS = 3

FACTORY = EnvFactory(OBSERVATION_MODE, world_mix=WORLD_MIX)

print("worlds:", [world.name for world in FACTORY.worlds])
print("num_envs:", NUM_ENVS_LIST)
print("warmup_steps:", WARMUP_STEPS)
print("measure_steps:", MEASURE_STEPS)
print("repeats:", REPEATS)

## Benchmark Helpers

In [ ]:
# cell: benchmark-helpers; requires: benchmark-config

def training_fns(factory, num_envs, params):
    if num_envs % len(factory.worlds):
        raise ValueError(f"num_envs must be divisible by {len(factory.worlds)}")
    slots_per_world = num_envs // len(factory.worlds)
    # Experimental notebook code: reuse the current implementation factories without changing production API yet.
    return [
        factory._training_factory(world, params)
        for world in factory.worlds
        for _ in range(slots_per_world)
    ]


def make_vector_env(kind, num_envs):
    if kind == "sync":
        return FACTORY.make_training_env(num_envs, params=PARAMS)
    if kind == "async":
        return AsyncVectorEnv(training_fns(FACTORY, num_envs, PARAMS))
    raise ValueError(f"unknown vector env kind: {kind}")


def sampled_actions(env, n_steps):
    return [env.action_space.sample() for _ in range(n_steps)]


def measure_once(kind, num_envs, repeat):
    env = make_vector_env(kind, num_envs)
    try:
        env.reset(seed=SEED + repeat)

        for actions in sampled_actions(env, WARMUP_STEPS):
            env.step(actions)

        actions_batch = sampled_actions(env, MEASURE_STEPS)
        started = time.perf_counter()
        for actions in actions_batch:
            env.step(actions)
        elapsed = time.perf_counter() - started

        env_steps = num_envs * MEASURE_STEPS
        return {
            "kind": kind,
            "num_envs": num_envs,
            "repeat": repeat,
            "status": "ok",
            "seconds": elapsed,
            "env_steps": env_steps,
            "env_steps_per_second": env_steps / elapsed,
            "error": "",
        }
    finally:
        env.close()

## Run Benchmark

In [ ]:
# cell: run-benchmark; requires: benchmark-helpers

rows = []

for num_envs in NUM_ENVS_LIST:
    for kind in ("sync", "async"):
        for repeat in range(REPEATS):
            try:
                row = measure_once(kind, num_envs, repeat)
                print(
                    f"{kind:5s} num_envs={num_envs:2d} repeat={repeat} "
                    f"{row['env_steps_per_second']:9.0f} env-steps/s "
                    f"({row['seconds']:.3f}s)"
                )
            except Exception as exc:
                row = {
                    "kind": kind,
                    "num_envs": num_envs,
                    "repeat": repeat,
                    "status": "fail",
                    "seconds": None,
                    "env_steps": num_envs * MEASURE_STEPS,
                    "env_steps_per_second": None,
                    "error": repr(exc),
                }
                print(f"{kind:5s} num_envs={num_envs:2d} repeat={repeat} FAIL: {row['error']}")
            rows.append(row)

## Results

In [ ]:
# cell: results; requires: run-benchmark

import pandas as pd

df = pd.DataFrame(rows)
display(df)

ok = df[df["status"] == "ok"].copy()
summary = (
    ok.groupby(["kind", "num_envs"], as_index=False)["env_steps_per_second"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
display(summary)

pivot = summary.pivot(index="num_envs", columns="kind", values="mean")
if {"sync", "async"}.issubset(pivot.columns):
    pivot["async_vs_sync"] = pivot["async"] / pivot["sync"]
display(pivot)

## Reading the Result

This measures only vectorized environment stepping. A L4 or A100 is printed in the setup cell for run documentation, but the GPU should not dominate this benchmark. If `async_vs_sync` is clearly above `1.0`, `AsyncVectorEnv` is a candidate for the trainer path. If it is near or below `1.0`, the subprocess overhead probably eats the CPU parallelism benefit for this setup.